# 2. Data storage: a lakehouse on one laptop

The concept note proposes Airflow, Spark, dbt and Delta Lake. Those are right at national scale. Today, this data is megabytes, so we keep the **design** (bronze, silver, gold) and start with the smallest **engine**: Parquet files and DuckDB.

| Layer | Contains | Rule |
| --- | --- | --- |
| Bronze | Raw downloads | Never edited, so we can always rebuild |
| Silver | One clean table per subject | Named columns, correct types, tested |
| Gold | One row per commune and season | What models read |

In [1]:
import json, os, sys
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ["IMBUTO_ROOT"] = str(ROOT)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": .25, "font.size": 10})
GREEN, GREY, BROWN = "#1f6e3f", "#9aa89f", "#8c510a"

In [2]:
import duckdb
from imbuto.lakehouse import build
print(build.run(ROOT / "data/bronze") if False else "Tables already built by `make data`")
def size(p): return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e6
for layer in ("bronze", "silver", "gold"):
    print(f"{layer:7}{size(ROOT / 'data' / layer):8.1f} MB")

Tables already built by `make data`
bronze    239.4 MB
silver      0.9 MB
gold        0.3 MB


## From a grid to a commune

Farmers think in communes, not grid cells. For each commune, we average the rainfall cells whose centres fall inside it. A very small commune may contain no cell centre, so it takes the nearest cell. The `cells` column records this, so nothing is hidden.

In [3]:
rain = pd.read_parquet(ROOT / "data/silver/rain_monthly.parquet")
print(f"{len(rain):,} rows, one per commune and month")
rain.groupby("commune_id").cells.first().describe()[["min", "50%", "max"]].to_frame("CHIRPS cells per commune")

65,331 rows, one per commune and month


,CHIRPS cells per commune
min,1.0
50%,6.0
max,18.0


## Query Parquet directly with SQL

There is no server to run or install. This is the same SQL a ministry analyst could use tomorrow. Here: which Septembers were driest across Burundi?

In [4]:
con = duckdb.connect()
con.execute(f"create view rain as select * from '{ROOT}/data/silver/rain_monthly.parquet'")
con.execute(f"create view communes as select * exclude (geojson) from '{ROOT}/data/silver/communes.parquet'")
con.sql("""
    select year(month) as year, round(avg(rain_mm)) as september_mm
    from rain where month(month) = 9
    group by 1 order by 2 limit 6
""").df()

,year,september_mm
0,1993,17.0
1,2000,24.0
2,1997,24.0
3,2002,25.0
4,1984,33.0
5,1992,33.0


## Seasons are the unit of farming

Burundi's main rain-fed seasons cross calendar boundaries: **Season A** runs from September into January, and **Season B** from February to May. So gold is built around seasons, not years.

Each gold row holds what is known at the issue date, plus the target that comes later. Keeping these apart in the table design is what stops leakage.

In [5]:
gold = pd.read_parquet(ROOT / "data/gold/season_features.parquet")
print(gold.groupby("season").year.agg(["min", "max", "count"]))
gold[gold.commune == "Bugarama"].tail(3)[["commune", "season", "year", "rain_so_far", "rain_prior_3m", "oni", "dmi", "target_rain"]]

         min   max  count
season                   
A       1981  2026   5474
B       1982  2026   5355


,commune,season,year,rain_so_far,rain_prior_3m,oni,dmi,target_rain
10514,Bugarama,B,2024,108.32,841.94,1.99,0.765,576.86
10633,Bugarama,B,2025,126.55,640.06,-0.43,-0.196,690.86
10752,Bugarama,B,2026,166.38,626.14,-0.60,0.123,624.51


## Data quality tests

A pipeline that runs is not the same as a pipeline that is right. These checks stop bad data before it reaches a model. In production, they would run on every refresh.

In [6]:
checks = {
    "119 communes present": gold.commune_id.nunique() == 119,
    "no negative rain": (rain.rain_mm >= 0).all(),
    "one row per commune, season and year": not gold.duplicated(["commune_id", "season", "year"]).any(),
    "only the unfinished season lacks a target": gold[gold.target_rain.isna()].groupby(["season", "year"]).ngroups <= 1,
    "September is wetter than July": rain[rain.month.dt.month == 9].rain_mm.mean() > rain[rain.month.dt.month == 7].rain_mm.mean(),
}
for name, ok in checks.items(): print("PASS" if ok else "FAIL", name)
assert all(checks.values())

PASS 119 communes present
PASS no negative rain
PASS one row per commune, season and year
PASS only the unfinished season lacks a target
PASS September is wetter than July


## A first useful product, with no model at all

Gold already answers a real question: **how does this season compare with each commune's own history?** This is measured, not predicted. Remember it, because in notebook 4 it becomes the thing we actually ship.

In [7]:
status = json.loads((ROOT / "artifacts/rain/rain_status.json").read_text())
table = pd.DataFrame(status["communes"])
print(f"Season {status['season']} {status['year']}, observed through {status['observed_through']}")
print("Median commune:", int(table.percent_of_normal.median()), "% of normal")
order = ["very_dry", "dry", "normal", "wet", "very_wet"]
table.status.value_counts().reindex(order, fill_value=0).rename("communes").to_frame()

Season A 2026, observed through 2026-09
Median commune: 59 % of normal


,communes
status,
very_dry,16
dry,61
normal,42
wet,0
very_wet,0


**When to move to Spark and Delta:** when data outgrows one machine, or many writers appear. Because the layers are the same, that changes the engine, not the design.

**Next:** notebook 3 trains the crop disease model.